# PolicyRec v1.1 — 공통 컬럼 정규화 파이프라인 (업데이트판)

## 이 노트북의 역할

v1.0이 "원본을 잃지 않는 것"에 집중했다면,
v1.1은 **추천·검색·필터에 실제로 쓸 수 있는 형태로 데이터를 정리하는 기준을 잡는 것**이 목적입니다.

## 이번 업데이트에서 반영한 수정사항

| # | 수정 내용 | 이유 |
| :--- | :--- | :--- |
| 1 | `USE_MOCK = Falseㅎ` 오타 수정 | 실행 오류 방지 |
| 2 | `region` / `provider` 컬럼 분리 | Bizinfo region에 부처명(과기정통부 등)이 섞여있어 지역 필터가 깨짐 |
| 3 | Youth `zipCd` → `region_code` 보존 + `region` 한글 광역명 추정 | 행정코드(`50110` 등)로는 검색 불가 |
| 4 | 날짜 "상시"/"추후공지" 처리 | 상시 = `2099-12-31`, 추후공지 = 문자열 그대로 |
| 5 | `target_age_min`/`max` 유효성 검사 추가 | min > max, 0/0 케이스 자동 보정 |
| 6 | `benefit_type` 컬럼 삭제 | 3개 source 전부 결측(100%) |
| 7 | `summary` = title + 원본summary 통합 | RAG 임베딩 품질 향상 |
| 8 | Youth summary = `plcyExplnCn` + `plcySprtCn` | 정책 설명 + 지원 내용 함께 포함 |
| 9 | K-Startup `source_id` float→str 변환 | `177322.0` → `"177322"` |

## 버전별 역할 요약

| 버전 | 목표 | 결과물 | 상태 |
| :--- | :--- | :--- | :--- |
| `v1.0` | API 원본 보존 및 출처 추적 | `combined_raw_columns.csv` | ✅ 완료 |
| **`v1.1`** | **공통 컬럼 정규화 + region/provider 분리** | **`combined_normalized_v1_1.csv`** | **🔄 이 노트북** |
| `v1.2` | SQLite DB + Chroma 임베딩 인덱스 | `policyrec.db`, `chroma/` | 🎯 다음 단계 |
| `v1.3` | 첨부파일(PDF/HWP) 본문 추가 + 시군구 매핑 정밀화 | | 예정 |
| `v1.4` | LLM (Gemma) 기반 RAG 추천 | | 예정 |

In [ ]:
# ============================================================
# 0. 기본 설정
# ============================================================
# 자주 바꿀 값은 이 셀에 모아 두었습니다.

from pathlib import Path
import re
import json
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(v): print(v)


# ============================================================
# 사용자가 자주 바꿀 설정값
# ============================================================

# True  : mock 데이터로 실행 (실제 CSV 없어도 동작)
# False : 실제 combined_raw_columns.csv 사용
USE_MOCK = False   # ← 오타 수정됨 (이전: Falseㅎ)

SOURCES = ["biz", "kst", "youth"]

SOURCE_LABELS = {
    "biz":   "Bizinfo",
    "kst":   "K-Startup",
    "youth": "Youthcenter",
}

CSV_ENCODING     = "utf-8-sig"
PREVIEW_ROW_COUNT = 3

# ============================================================
# 결측값 처리 기준
# ============================================================
# 아래 값들은 팀 논의를 통해 변경할 수 있습니다.

DEFAULT_REGION   = "전국"       # region 없을 때
DEFAULT_PROVIDER = ""           # provider 없을 때 (빈 문자열)
DEFAULT_AGE_MIN  = 0            # 연령 하한 없을 때
DEFAULT_AGE_MAX  = 99           # 연령 상한 없을 때
DEFAULT_CATEGORY = "기타"       # category 없을 때

# ============================================================
# 특수 날짜 값 처리
# ============================================================
# "상시", "수시" 등 마감일 없는 공고는 먼 미래 날짜(2099-12-31)로 변환
# → SQLite에서 end_date 기반 필터링이 깨지지 않도록
# "추후공지", "미정"은 그대로 문자열 보존
# → 사용자 UI에서 "마감일 미정"으로 표시할 수 있도록

PERPETUAL_DATE = "2099-12-31"
PERPETUAL_KEYWORDS = ["상시", "상시모집", "수시", "연중", "상시접수"]
TBD_KEYWORDS       = ["추후공지", "추후 공지", "미정", "별도공지", "추후안내"]

# ============================================================
# 경로 설정
# ============================================================

PROJECT_ROOT    = Path.cwd()
CLEAN_ROOT      = PROJECT_ROOT / "data" / "clean"
RAW_MERGED_FILE = CLEAN_ROOT / "combined_raw_columns.csv"       # v1.0 결과 (입력)
NORMALIZED_FILE = CLEAN_ROOT / "combined_normalized_v1_1.csv"   # v1.1 결과 (출력)

print("설정 완료")
print(f"  USE_MOCK     : {USE_MOCK}")
print(f"  입력 파일    : {RAW_MERGED_FILE}")
print(f"  출력 파일    : {NORMALIZED_FILE}")

## 공통 스키마 정의

### 최종 공통 컬럼 (v1.1 업데이트)

| 컬럼명 | 설명 | 비고 |
| :--- | :--- | :--- |
| `source` | 출처 코드 | biz / kst / youth |
| `source_id` | 원본 고유 ID | K-Startup은 float → int → str 변환 |
| `title` | 공고/정책 제목 | |
| `summary` | 통합 요약 (title + 원본summary) | RAG 임베딩용 확장 텍스트 |
| `category` | 분야 | 없으면 `기타` |
| `region` | 지역 | **지역명만** (전국 포함). 없으면 `전국` |
| `region_code` 🆕 | 원본 지역 코드 | Youth zipCd 보존용 (예: `50110,50130`) |
| `provider` 🆕 | 주관/집행 기관 | 중앙부처, 지자체 기관, 운영기관 등 |
| `target_group` | 대상 | |
| `target_age_min` | 최소 연령 | 없으면 `0`, 유효성 검증됨 |
| `target_age_max` | 최대 연령 | 없으면 `99`, 유효성 검증됨 |
| `start_date` | 신청 시작일 | `YYYY-MM-DD` 또는 `추후공지` |
| `end_date` | 신청 마감일 | `YYYY-MM-DD` (상시 = `2099-12-31`) 또는 `추후공지` |
| `detail_url` | 상세 URL | |

### 삭제된 컬럼

- ❌ `benefit_type` : 3개 source 모두 대응 컬럼 없어 100% 결측 → 제거

### source별 원본 컬럼 → 공통 컬럼 매핑

| 공통 컬럼 | Bizinfo | K-Startup | Youthcenter |
| :--- | :--- | :--- | :--- |
| `source_id` | `pblancId` | `pbanc_sn` *(float→int→str)* | `plcyNo` |
| `title` | `pblancNm` | `biz_pbanc_nm` | `plcyNm` |
| `summary` | title + `bsnsSumryCn` | title + `pbanc_ctnt` | title + `plcyExplnCn` + `plcySprtCn` |
| `category` | `pldirSportRealmLclasCodeNm` | `supt_biz_clsfc` | `lclsfNm` |
| `region` | `jrsdInsttNm` *(지역명인 경우만)* | `supt_regin` | `zipCd`를 광역명으로 추정 |
| `region_code` | — | — | `zipCd` 원본 보존 |
| `provider` | `excInsttNm` 또는 `jrsdInsttNm` *(부처명인 경우)* | `pbanc_ntrp_nm` 또는 `biz_prch_dprt_nm` | `sprvsnInstCdNm` 또는 `operInstCdNm` |
| `target_group` | `trgetNm` | `aply_trgt` | `ptcpPrpTrgtCn` |
| `target_age` | ❌ → 0~99 | `biz_trgt_age` (파싱 필요) | min/max 컬럼 분리 |
| `start_date` | `reqstBeginEndDe` (분리) | `pbanc_rcpt_bgng_dt` | `bizPrdBgngYmd` |
| `end_date` | `reqstBeginEndDe` (분리) | `pbanc_rcpt_end_dt` | `bizPrdEndYmd` |
| `detail_url` | `pblancUrl` | `detl_pg_url` | `aplyUrlAddr` |

In [ ]:
# ============================================================
# 공통 스키마 및 매핑 테이블
# ============================================================

COMMON_COLUMNS = [
    "source", "source_id", "title", "summary", "category",
    "region", "region_code", "provider",
    "target_group", "target_age_min", "target_age_max",
    "start_date", "end_date", "detail_url",
]

# 원본 컬럼명 → 공통 컬럼명 매핑
# None : 해당 source에 대응 컬럼 없음 → 기본값 처리
# list : 여러 후보 중 먼저 존재하는 컬럼 사용 (방어적 처리)
COLUMN_MAP = {
    "biz": {
        "source_id":       "pblancId",
        "title":           "pblancNm",
        "summary_main":    "bsnsSumryCn",      # title과 합쳐져 summary로
        "category":        "pldirSportRealmLclasCodeNm",
        "region_raw":      "jrsdInsttNm",       # 지역/부처 판별 필요
        "provider_alt":    "excInsttNm",        # 집행기관 (지역 케이스일 때 provider)
        "target_group":    "trgetNm",
        "target_age":      None,                # 없음 → 0~99
        "date_range":      "reqstBeginEndDe",   # "YYYY-MM-DD ~ YYYY-MM-DD" 분리 필요
        "detail_url":      "pblancUrl",
    },
    "kst": {
        "source_id":       "pbanc_sn",          # float → int → str 변환 필요
        "title":           "biz_pbanc_nm",
        "summary_main":    "pbanc_ctnt",
        "category":        "supt_biz_clsfc",
        "region_raw":      "supt_regin",
        "provider_alts":   ["pbanc_ntrp_nm", "biz_prch_dprt_nm"],  # 후보들
        "target_group":    "aply_trgt",
        "target_age":      "biz_trgt_age",      # "만 19~39세" 텍스트 파싱
        "start_date":      "pbanc_rcpt_bgng_dt",
        "end_date":        "pbanc_rcpt_end_dt",
        "detail_url":      "detl_pg_url",
    },
    "youth": {
        "source_id":       "plcyNo",
        "title":           "plcyNm",
        "summary_main":    "plcyExplnCn",       # 정책 설명
        "summary_extra":   "plcySprtCn",        # 정책 지원 내용 (추가 결합)
        "category":        "lclsfNm",
        "region_raw":      "zipCd",              # 행정코드 (예: "50110,50130")
        "provider_alts":   ["sprvsnInstCdNm", "operInstCdNm"],  # 주관/운영 기관
        "target_group":    "ptcpPrpTrgtCn",
        "target_age":      "_age_range",        # min/max 컬럼 자동 탐색
        "start_date":      "bizPrdBgngYmd",
        "end_date":        "bizPrdEndYmd",
        "detail_url":      "aplyUrlAddr",
    },
}

print("스키마 정의 완료")
print("공통 컬럼:", COMMON_COLUMNS)
print(f"총 {len(COMMON_COLUMNS)}개 컬럼")

## 지역 판별 및 행정코드 매핑 정의

### region / provider 판별 로직

Bizinfo의 `jrsdInsttNm` 컬럼에는 **지역명과 중앙부처명이 혼재**합니다.

예시:
- 지역명 케이스: `경상북도`, `서울특별시`, `충청북도`
- 부처명 케이스: `과학기술정보통신부`, `기후에너지환경부`, `고용노동부`

따라서 문자열에 **17개 광역지자체 키워드**가 포함되어 있는지로 판별합니다.

| 입력 | 판별 결과 | region | provider |
| :--- | :--- | :--- | :--- |
| `경상북도` | 지역 | `경상북도` | `excInsttNm`값 |
| `과학기술정보통신부` | 부처 | `전국` | `과학기술정보통신부` |
| `서울특별시 강남구` | 지역 | `서울특별시` | `excInsttNm`값 |
| (None) | 없음 | `전국` | `""` |

### Youth zipCd 광역코드 매핑

대한민국 행정표준코드의 **앞 2자리**는 광역지자체를 구분합니다.

| 코드 | 광역지자체 | 코드 | 광역지자체 |
| :--- | :--- | :--- | :--- |
| `11` | 서울특별시 | `41` | 경기도 |
| `26` | 부산광역시 | `42` | 강원특별자치도 |
| `27` | 대구광역시 | `43` | 충청북도 |
| `28` | 인천광역시 | `44` | 충청남도 |
| `29` | 광주광역시 | `45` | 전북특별자치도 |
| `30` | 대전광역시 | `46` | 전라남도 |
| `31` | 울산광역시 | `47` | 경상북도 |
| `36` | 세종특별자치시 | `48` | 경상남도 |
| | | `50` | 제주특별자치도 |

**처리 방식 (하이브리드):**
1. `region_code` 컬럼: 원본 zipCd 그대로 보존 (예: `"50110,50130"`)
2. `region` 컬럼: 앞 2자리로 광역 추정 (예: `"제주특별자치도"`)
3. 여러 광역이 섞이면 쉼표로 병기 (예: `"서울특별시, 경기도"`)

**한계 (v1.3 과제):**
- 시군구 단위 필터링은 `region_code`를 별도 파싱해야 함
- 현 매핑은 2024~2026년 기준 (행정개편 시 업데이트 필요)

In [ ]:
# ============================================================
# 17개 광역지자체 키워드 및 행정코드 매핑
# ============================================================

# 지역명 판별용 키워드 (긴 형태 + 짧은 형태)
# 긴 형태부터 체크해야 "서울특별시"가 "서울"보다 먼저 매칭됨
REGION_KEYWORDS_LONG = [
    "서울특별시", "부산광역시", "대구광역시", "인천광역시",
    "광주광역시", "대전광역시", "울산광역시", "세종특별자치시",
    "경기도", "강원특별자치도", "강원도",
    "충청북도", "충청남도",
    "전북특별자치도", "전라북도", "전라남도",
    "경상북도", "경상남도",
    "제주특별자치도", "제주도",
]

# 짧은 형태 → 정규화된 긴 형태 매핑
REGION_SHORT_TO_LONG = {
    "서울": "서울특별시", "부산": "부산광역시", "대구": "대구광역시",
    "인천": "인천광역시", "광주": "광주광역시", "대전": "대전광역시",
    "울산": "울산광역시", "세종": "세종특별자치시",
    "경기": "경기도", "강원": "강원특별자치도",
    "충북": "충청북도", "충남": "충청남도",
    "전북": "전북특별자치도", "전남": "전라남도",
    "경북": "경상북도", "경남": "경상남도",
    "제주": "제주특별자치도",
}

# 행정표준코드 앞 2자리 → 광역지자체
REGION_CODE_TO_NAME = {
    "11": "서울특별시", "26": "부산광역시", "27": "대구광역시",
    "28": "인천광역시", "29": "광주광역시", "30": "대전광역시",
    "31": "울산광역시", "36": "세종특별자치시",
    "41": "경기도", "42": "강원특별자치도", "43": "충청북도",
    "44": "충청남도", "45": "전북특별자치도", "46": "전라남도",
    "47": "경상북도", "48": "경상남도", "50": "제주특별자치도",
}


def classify_region_or_provider(text):
    """
    문자열이 지역명인지 기관명(부처 등)인지 판별합니다.

    반환값: (region, provider)
      - 지역명으로 판별 → (정규화된 지역명, None)
      - 기관명으로 판별 → (None, 원본 문자열)
      - 빈 값             → (None, None)

    예시:
      "경상북도"           → ("경상북도", None)
      "서울특별시 강남구"  → ("서울특별시", None)
      "과학기술정보통신부" → (None, "과학기술정보통신부")
      "전국"               → ("전국", None)
      None                 → (None, None)
    """
    if text is None or pd.isna(text) or str(text).strip() == "":
        return None, None

    s = str(text).strip()

    # "전국"은 특수 케이스 (region으로 취급)
    if s == "전국":
        return "전국", None

    # 긴 형태 매칭 시도 (서울특별시가 서울보다 먼저)
    for long_name in REGION_KEYWORDS_LONG:
        if long_name in s:
            return long_name, None

    # 짧은 형태 매칭 시도 (단, "부"로 끝나는 부처명은 제외)
    # 예: "산업통상부"의 "통상"이 실수로 지역 매칭 안 되도록
    if not s.endswith(("부", "청", "처", "위원회", "공단", "진흥원", "센터")):
        for short_name, long_name in REGION_SHORT_TO_LONG.items():
            if short_name in s:
                return long_name, None

    # 지역명 아님 → provider로 취급
    return None, s


def zipcd_to_region(zipcd):
    """
    Youth zipCd 행정코드를 광역지자체 한글명으로 변환합니다.
    원본 zipCd는 region_code 컬럼에 그대로 보존합니다.

    입력 예시:
      "50110"           → "제주특별자치도"
      "50110,50130"     → "제주특별자치도"             (모두 제주)
      "11110,41110"     → "서울특별시, 경기도"         (여러 광역)
      "99999"           → DEFAULT_REGION (매핑 실패)
      None              → DEFAULT_REGION
    """
    if zipcd is None or pd.isna(zipcd) or str(zipcd).strip() == "":
        return DEFAULT_REGION

    s = str(zipcd).strip()
    codes = [c.strip() for c in s.split(",") if c.strip()]

    regions = []
    for code in codes:
        # 행정코드는 보통 5~10자리 숫자. 앞 2자리가 광역 코드.
        if len(code) >= 2 and code[:2] in REGION_CODE_TO_NAME:
            region = REGION_CODE_TO_NAME[code[:2]]
            if region not in regions:
                regions.append(region)

    if not regions:
        return DEFAULT_REGION

    return ", ".join(regions)


# ============================================================
# 간단 셀프 테스트
# ============================================================

_test_cases_region = [
    ("경상북도",           "경상북도",       None),
    ("서울특별시 강남구",   "서울특별시",     None),
    ("과학기술정보통신부",  None,             "과학기술정보통신부"),
    ("기후에너지환경부",    None,             "기후에너지환경부"),
    ("고용노동부",          None,             "고용노동부"),
    ("산업통상부",          None,             "산업통상부"),
    ("전국",                "전국",           None),
    (None,                  None,             None),
    ("",                    None,             None),
]
print("=== classify_region_or_provider 테스트 ===")
for inp, exp_r, exp_p in _test_cases_region:
    got_r, got_p = classify_region_or_provider(inp)
    status = "✓" if (got_r == exp_r and got_p == exp_p) else "✗"
    print(f"  {status} {inp!r:30} → region={got_r!r}, provider={got_p!r}")

_test_cases_zip = [
    ("50110",         "제주특별자치도"),
    ("50110,50130",   "제주특별자치도"),
    ("11110",         "서울특별시"),
    ("11110,41110",   "서울특별시, 경기도"),
    ("99999",         DEFAULT_REGION),
    (None,            DEFAULT_REGION),
]
print("\n=== zipcd_to_region 테스트 ===")
for inp, exp in _test_cases_zip:
    got = zipcd_to_region(inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!r:20} → {got!r}")

## Mock 데이터 정의

실제 API 응답이 없는 상태에서 함수 동작을 검증하기 위한 샘플 데이터입니다.

각 source의 **실제 API 응답 구조와 컬럼명**을 그대로 모방했습니다.
`USE_MOCK = False`이면 이 셀은 실행만 하고 값은 사용하지 않습니다.

> ⚠️ **이번 버전에서 추가한 Mock 케이스**
> - Bizinfo: region에 부처명(`과학기술정보통신부`)이 들어간 케이스
> - Bizinfo: `excInsttNm` 컬럼 추가 (집행기관)
> - K-Startup: `pbanc_sn`을 float로 넣어 소수점 변환 테스트
> - Youth: `zipCd`를 행정코드(`50110,50130`)로 넣어 광역 매핑 테스트
> - Youth: `plcySprtCn` (지원 내용) 추가
> - 날짜: `상시`, `추후공지` 특수값 포함
> - 연령: `min > max` 역전 케이스 포함

In [ ]:
# ============================================================
# Mock 데이터 (USE_MOCK = True일 때만 사용)
# ============================================================

MOCK_ROWS = [
    # ── Bizinfo ──────────────────────────────────────────────
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "pblancId": "BIZ001",
        "pblancNm": "2026년 소상공인 경영안정자금 지원",
        "bsnsSumryCn": "소상공인의 경영 안정을 위한 저금리 융자 지원 사업입니다.",
        "pldirSportRealmLclasCodeNm": "금융",
        "jrsdInsttNm": "중소벤처기업부",                    # 부처명 케이스
        "excInsttNm": "소상공인시장진흥공단",                # 실제 집행 기관
        "trgetNm": "소상공인",
        "reqstBeginEndDe": "2026-04-01 ~ 2026-05-31",
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ001",
    },
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "pblancId": "BIZ002",
        "pblancNm": "[경북] 포항시 2026년 중소기업 인증획득 지원사업",
        "bsnsSumryCn": "포항시 중소기업 신뢰도 제고를 위한 인증 비용 지원.",
        "pldirSportRealmLclasCodeNm": None,                  # category 없음 → 기타
        "jrsdInsttNm": "경상북도",                           # 지역명 케이스
        "excInsttNm": "포항테크노파크",
        "trgetNm": "중소기업",
        "reqstBeginEndDe": "2026.05.01~2026.06.30",         # 점 구분자
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ002",
    },
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "pblancId": "BIZ003",
        "pblancNm": "2026년 우수 정보보호기술 지정제도 접수 공고",
        "bsnsSumryCn": "정보보호 기술 지정 제도 신청을 접수합니다.",
        "pldirSportRealmLclasCodeNm": "기술",
        "jrsdInsttNm": "과학기술정보통신부",                 # 부처명 케이스
        "excInsttNm": None,                                  # 집행기관 없음 → 주관기관 fallback
        "trgetNm": "창업벤처",
        "reqstBeginEndDe": "상시",                           # 특수값: 상시
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ003",
    },

    # ── K-Startup ─────────────────────────────────────────────
    {
        "_source": "kst", "_source_name": "K-Startup",
        "pbanc_sn": 177322.0,                                # float → str 변환 테스트
        "biz_pbanc_nm": "초기창업패키지",
        "pbanc_ctnt": "창업 3년 이내 초기 창업기업 대상 사업화 자금 지원.",
        "supt_biz_clsfc": "창업",
        "supt_regin": "전국",
        "pbanc_ntrp_nm": "창업진흥원",
        "biz_prch_dprt_nm": "초기창업팀",
        "aply_trgt": "창업 3년 이내 기업",
        "biz_trgt_age": "만 19~39세",
        "pbanc_rcpt_bgng_dt": "2026-04-10",
        "pbanc_rcpt_end_dt": "2026-05-10",
        "detl_pg_url": "https://www.k-startup.go.kr/KST001",
    },
    {
        "_source": "kst", "_source_name": "K-Startup",
        "pbanc_sn": 177319.0,
        "biz_pbanc_nm": "예비창업패키지",
        "pbanc_ctnt": "혁신 기술 창업 아이디어 보유 예비창업자 지원.",
        "supt_biz_clsfc": "창업",
        "supt_regin": "서울",
        "pbanc_ntrp_nm": "서울창조경제혁신센터",
        "biz_prch_dprt_nm": None,
        "aply_trgt": "예비창업자",
        "biz_trgt_age": "39세 이하",
        "pbanc_rcpt_bgng_dt": "20260415",                    # 8자리 숫자
        "pbanc_rcpt_end_dt": "추후공지",                     # 특수값: 추후공지
        "detl_pg_url": "https://www.k-startup.go.kr/KST002",
    },
    {
        "_source": "kst", "_source_name": "K-Startup",
        "pbanc_sn": 177313.0,
        "biz_pbanc_nm": "글로벌 액셀러레이팅",
        "pbanc_ctnt": "해외 진출 스타트업 글로벌 네트워킹 지원.",
        "supt_biz_clsfc": "글로벌",
        "supt_regin": None,                                  # → 전국
        "pbanc_ntrp_nm": None,
        "biz_prch_dprt_nm": None,
        "aply_trgt": "스타트업",
        "biz_trgt_age": "제한 없음",
        "pbanc_rcpt_bgng_dt": "2026-05-01",
        "pbanc_rcpt_end_dt": "2026-06-01",
        "detl_pg_url": "https://www.k-startup.go.kr/KST003",
    },

    # ── Youthcenter ───────────────────────────────────────────
    {
        "_source": "youth", "_source_name": "Youthcenter",
        "plcyNo": "YTH001",
        "plcyNm": "제주시 청년농업인 영농정착 지원",
        "plcyExplnCn": "영농초기 청년농업인에게 정착지원금 지급.",
        "plcySprtCn": "월 100만원 영농정착지원금 지원, 영농 교육, 컨설팅 제공.",   # 추가 결합 테스트
        "lclsfNm": "일자리",
        "zipCd": "50110,50130",                              # 제주 행정코드
        "sprvsnInstCdNm": "제주특별자치도",
        "operInstCdNm": "제주시농업기술센터",
        "ptcpPrpTrgtCn": "만 19~39세 청년농업인",
        "ageMin": "19",
        "ageMax": "39",
        "bizPrdBgngYmd": "20260401",
        "bizPrdEndYmd": "20261231",
        "aplyUrlAddr": "https://plus.gov.kr/YTH001",
    },
    {
        "_source": "youth", "_source_name": "Youthcenter",
        "plcyNo": "YTH002",
        "plcyNm": "3만원 주택",
        "plcyExplnCn": "신혼부부·자녀출산 가구 주거비 부담 완화.",
        "plcySprtCn": None,                                  # 지원 내용 없음
        "lclsfNm": "주거",
        "zipCd": "50110,50130",
        "sprvsnInstCdNm": "제주특별자치도",
        "operInstCdNm": None,
        "ptcpPrpTrgtCn": None,
        "ageMin": "0",                                       # 역전/결측 케이스
        "ageMax": "0",                                       # min==max==0 → 유효성 보정
        "bizPrdBgngYmd": "2026-01-01",
        "bizPrdEndYmd": "2026-12-31",
        "aplyUrlAddr": "https://plus.gov.kr/YTH002",
    },
    {
        "_source": "youth", "_source_name": "Youthcenter",
        "plcyNo": "YTH003",
        "plcyNm": "청년 내일저축계좌",
        "plcyExplnCn": "저소득 청년의 자산 형성 지원.",
        "plcySprtCn": "3년간 정부 매칭 저축.",
        "lclsfNm": "금융",
        "zipCd": None,                                        # → 전국
        "sprvsnInstCdNm": "보건복지부",
        "operInstCdNm": None,
        "ptcpPrpTrgtCn": "일하는 저소득 청년",
        "ageMin": None,
        "ageMax": "34",
        "bizPrdBgngYmd": "2026-05-02",
        "bizPrdEndYmd": "2026-05-31",
        "aplyUrlAddr": "https://plus.gov.kr/YTH003",
    },
]

print(f"mock 데이터 준비 완료: {len(MOCK_ROWS)}건")
print("source별:", {s: sum(1 for r in MOCK_ROWS if r['_source']==s) for s in SOURCES})

## Helper 함수 정의

| 함수명 | 역할 |
| :--- | :--- |
| `normalize_date()` | 다양한 날짜 형식 → `YYYY-MM-DD` 또는 특수값 |
| `parse_biz_date()` | Bizinfo `reqstBeginEndDe` (범위값) → start / end 분리 |
| `parse_kst_age()` | K-Startup `biz_trgt_age` (텍스트) → min / max 숫자 |
| `parse_youth_age()` | Youthcenter min/max 컬럼 → 숫자 추출 |
| `validate_age_range()` 🆕 | min > max, 0/0 케이스 자동 보정 |
| `fix_kst_source_id()` 🆕 | K-Startup source_id float → int → str |
| `pick_first_available()` 🆕 | 여러 후보 컬럼 중 먼저 값이 있는 것 선택 |
| `build_summary()` 🆕 | title + 원본summary 통합 (RAG 임베딩용) |
| `clean_bizinfo()`, `clean_kst()`, `clean_youth()` | source별 정규화 메인 함수 |

In [ ]:
# ============================================================
# 날짜 변환 함수
# ============================================================

def normalize_date(value):
    """
    다양한 날짜 형식을 표준값으로 변환합니다.

    변환 규칙:
      "20260420"       → "2026-04-20"      (8자리 숫자)
      "2026-04-20"     → "2026-04-20"      (이미 올바른 형식)
      "2026.04.20"     → "2026-04-20"      (점 구분자)
      "2026/04/20"     → "2026-04-20"      (슬래시 구분자)
      "상시", "수시"   → "2099-12-31"      (마감 없음 → 먼 미래)
      "추후공지"       → "추후공지"         (문자열 그대로)
      None / 빈값      → None

    ※ Bizinfo 범위값("YYYY-MM-DD ~ YYYY-MM-DD")은
      parse_biz_date()에서 먼저 분리한 뒤 이 함수로 전달합니다.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None

    s = str(value).strip()

    # 특수값 처리: 상시/수시 → 먼 미래
    for kw in PERPETUAL_KEYWORDS:
        if kw in s:
            return PERPETUAL_DATE

    # 특수값 처리: 추후공지 → 문자열 그대로
    for kw in TBD_KEYWORDS:
        if kw in s:
            return "추후공지"

    # 8자리 숫자: YYYYMMDD
    if re.fullmatch(r"\d{8}", s):
        return f"{s[:4]}-{s[4:6]}-{s[6:8]}"

    # 점·슬래시 → 하이픈
    s2 = re.sub(r"[./]", "-", s)

    # YYYY-MM-DD 형식 최종 확인
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s2):
        return s2

    # 알 수 없는 형식 → None (나중에 결측으로 처리됨)
    return None


def parse_biz_date(value):
    """
    Bizinfo reqstBeginEndDe를 (start_date, end_date) 튜플로 분리합니다.

    입력 예시:
      "2026-04-01 ~ 2026-05-31"  → ("2026-04-01", "2026-05-31")
      "2026.05.01~2026.06.30"    → ("2026-05-01", "2026-06-30")
      "상시"                      → (None, "2099-12-31")   # 시작은 미정, 마감만 상시
      "추후공지"                  → (None, "추후공지")
      None                        → (None, None)
    """
    if pd.isna(value) or str(value).strip() == "":
        return None, None

    s = str(value).strip()

    # 특수값이 단독으로 들어온 경우 (범위가 아님)
    for kw in PERPETUAL_KEYWORDS:
        if kw in s and "~" not in s:
            return None, PERPETUAL_DATE
    for kw in TBD_KEYWORDS:
        if kw in s and "~" not in s:
            return None, "추후공지"

    # 범위값 분리
    parts = re.split(r"\s*~\s*", s)
    start = normalize_date(parts[0]) if len(parts) >= 1 else None
    end   = normalize_date(parts[1]) if len(parts) >= 2 else None
    return start, end


# ============================================================
# 연령 파싱 함수
# ============================================================

def parse_kst_age(value):
    """
    K-Startup biz_trgt_age 텍스트에서 (min, max) 숫자 튜플을 추출합니다.

    입력 예시:
      "만 19~39세"  → (19, 39)
      "39세 이하"   → (0,  39)
      "19세 이상"   → (19, 99)
      "제한 없음"   → (0,  99)
      None          → (0,  99)
    """
    if pd.isna(value) or str(value).strip() == "":
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    s = str(value).strip()

    m = re.search(r"(\d+)\s*~\s*(\d+)", s)
    if m:
        return int(m.group(1)), int(m.group(2))

    m = re.search(r"(\d+)\s*세?\s*이하", s)
    if m:
        return DEFAULT_AGE_MIN, int(m.group(1))

    m = re.search(r"(\d+)\s*세?\s*이상", s)
    if m:
        return int(m.group(1)), DEFAULT_AGE_MAX

    return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX


def parse_youth_age(min_val, max_val):
    """
    Youthcenter ageMin / ageMax 컬럼에서 숫자를 추출합니다.

    입력 예시:
      ("19", "34")    → (19, 34)
      (None, "34")    → (0,  34)
      (None, None)    → (0,  99)
    """
    try:
        mn = int(float(min_val)) if not pd.isna(min_val) else DEFAULT_AGE_MIN
    except (ValueError, TypeError):
        mn = DEFAULT_AGE_MIN

    try:
        mx = int(float(max_val)) if not pd.isna(max_val) else DEFAULT_AGE_MAX
    except (ValueError, TypeError):
        mx = DEFAULT_AGE_MAX

    return mn, mx


def validate_age_range(age_min, age_max):
    """
    target_age_min / max 유효성 검사 및 보정.

    처리 규칙:
      - min > max         → 둘 다 기본값 (0, 99)
      - min == max == 0   → 기본값 (0, 99)  (youth의 결측 케이스)
      - 음수              → 기본값
      - 비정상 큰 값      → 기본값
      - 정상              → 그대로

    이 함수는 '의미상 이상한 범위'를 감지해서 사용자가 필터링할 때
    공고가 이상하게 누락되는 것을 방지합니다.
    """
    try:
        mn = int(age_min)
        mx = int(age_max)
    except (ValueError, TypeError):
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    # 음수 또는 비정상 범위
    if mn < 0 or mx < 0 or mn > 120 or mx > 120:
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    # min > max 역전
    if mn > mx:
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    # min == max == 0 (결측 의심 케이스)
    if mn == 0 and mx == 0:
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    return mn, mx


# ============================================================
# source_id 및 기타 helper
# ============================================================

def fix_kst_source_id(value):
    """
    K-Startup pbanc_sn은 CSV 로드 시 float로 읽혀 '177322.0' 형태가 됨.
    이를 '177322' 정수 문자열로 변환.

    입력 예시:
      "177322.0"  → "177322"
      177322.0    → "177322"
      "177322"    → "177322"   (이미 깔끔)
      None        → None
    """
    if value is None or pd.isna(value):
        return None
    try:
        # float 경유하면 '.0' 자동 제거됨
        return str(int(float(value)))
    except (ValueError, TypeError):
        # 숫자 변환 실패 시 원본 문자열 그대로
        return str(value).strip()


def pick_first_available(df, col_candidates):
    """
    여러 후보 컬럼 중 '존재하고 non-null 값이 가장 많은' 컬럼의 Series를 반환.
    방어적 처리: 후보 중 하나도 없으면 빈 Series 반환.

    입력:
      df              : DataFrame
      col_candidates  : 컬럼명 리스트 (예: ["excInsttNm", "jrsdInsttNm"])

    반환: pd.Series (길이 = len(df))
    """
    for col in col_candidates:
        if col in df.columns:
            return df[col]
    # 후보 중 아무 것도 없으면 None 채움
    return pd.Series([None] * len(df))


def build_summary(title, main, extra=None):
    """
    title + 원본summary + (선택) 추가 텍스트를 하나의 summary로 통합.
    RAG 임베딩 품질을 높이기 위해 제목을 함께 포함합니다.

    포맷: "{title}\n\n{main}\n\n{extra}"

    None/빈값은 건너뛰고, 실제 값이 있는 것만 \n\n으로 연결.
    모두 비어있으면 빈 문자열 반환.
    """
    parts = []
    for piece in (title, main, extra):
        if piece is not None and not pd.isna(piece):
            s = str(piece).strip()
            if s and s.lower() not in ("nan", "none"):
                parts.append(s)
    return "\n\n".join(parts)


# ============================================================
# 셀프 테스트
# ============================================================

print("=== validate_age_range 테스트 ===")
_age_tests = [
    ((19, 39), (19, 39)),     # 정상
    ((39, 19), (0, 99)),      # 역전
    ((0, 0),   (0, 99)),      # 0/0 케이스
    ((-5, 30), (0, 99)),      # 음수
    ((0, 200), (0, 99)),      # 비정상 큰 값
    (("19", "39"), (19, 39)), # 문자열 숫자
    ((None, 30), (0, 99)),    # None
]
for inp, exp in _age_tests:
    got = validate_age_range(*inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!s:20} → {got}")

print("\n=== fix_kst_source_id 테스트 ===")
for inp, exp in [("177322.0", "177322"), (177322.0, "177322"),
                 ("177322", "177322"), (None, None)]:
    got = fix_kst_source_id(inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!r:15} → {got!r}")

print("\n=== normalize_date 특수값 테스트 ===")
for inp, exp in [("상시", "2099-12-31"), ("수시모집", "2099-12-31"),
                 ("추후공지", "추후공지"), ("미정", "추후공지"),
                 ("20260401", "2026-04-01"), ("2026-04-01", "2026-04-01"),
                 (None, None)]:
    got = normalize_date(inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!r:15} → {got!r}")

print("\n날짜·연령·기타 helper 함수 정의 완료")

## source별 정규화 함수

각 source의 원본 컬럼을 공통 스키마로 변환합니다.

### region / provider 분리 방식

- **Bizinfo**: `jrsdInsttNm`을 `classify_region_or_provider()`로 분류
  - 지역명이면 → `region` = 값, `provider` = `excInsttNm` (집행기관)
  - 부처명이면 → `region` = `"전국"`, `provider` = `jrsdInsttNm`
- **K-Startup**: `supt_regin`은 이미 지역명으로 잘 정리됨 → `region`에 그대로, `provider`는 `pbanc_ntrp_nm` 또는 `biz_prch_dprt_nm`
- **Youth**: `zipCd`를 `region_code`에 보존 + `zipcd_to_region()`으로 광역명 추정 → `region`, `provider`는 `sprvsnInstCdNm` 또는 `operInstCdNm`

In [ ]:
# ============================================================
# source별 정규화 함수
# ============================================================

def clean_bizinfo(df):
    """
    Bizinfo DataFrame을 공통 스키마로 변환합니다.

    주요 처리:
    - reqstBeginEndDe : 범위값 → start_date / end_date 분리 (상시/추후공지 포함)
    - jrsdInsttNm     : 지역명/부처명 판별 → region / provider 분리
    - summary         : title + bsnsSumryCn 통합
    - target_age      : 컬럼 없음 → 0 / 99 기본값
    """
    m   = COLUMN_MAP["biz"]
    out = pd.DataFrame()

    out["source"]    = df["_source"]
    out["source_id"] = df.get(m["source_id"])
    out["title"]     = df.get(m["title"])

    # summary = title + 원본 요약
    titles   = df.get(m["title"], pd.Series([None]*len(df)))
    summaries = df.get(m["summary_main"], pd.Series([None]*len(df)))
    out["summary"] = [build_summary(t, s) for t, s in zip(titles, summaries)]

    out["category"] = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)

    # region / provider 분리
    region_raw = df.get(m["region_raw"], pd.Series([None]*len(df)))
    provider_alt = df.get(m["provider_alt"], pd.Series([None]*len(df)))  # excInsttNm

    regions = []
    providers = []
    for raw, alt in zip(region_raw, provider_alt):
        r, p = classify_region_or_provider(raw)
        if r is not None:
            # 지역명 케이스 → provider는 excInsttNm 사용 (없으면 원본 jrsdInsttNm fallback)
            regions.append(r)
            if alt is not None and not pd.isna(alt) and str(alt).strip():
                providers.append(str(alt).strip())
            else:
                providers.append(str(raw).strip() if raw is not None and not pd.isna(raw) else DEFAULT_PROVIDER)
        elif p is not None:
            # 부처명 케이스 → region은 전국
            regions.append(DEFAULT_REGION)
            providers.append(p)
        else:
            # 둘 다 None → 기본값
            regions.append(DEFAULT_REGION)
            providers.append(DEFAULT_PROVIDER)

    out["region"]      = regions
    out["region_code"] = None   # Bizinfo는 코드값 없음
    out["provider"]    = providers

    out["target_group"]   = df.get(m["target_group"])
    out["target_age_min"] = DEFAULT_AGE_MIN
    out["target_age_max"] = DEFAULT_AGE_MAX

    # 날짜 범위 분리
    date_col = m["date_range"]
    if date_col in df.columns:
        parsed = df[date_col].apply(parse_biz_date)
        out["start_date"] = parsed.apply(lambda t: t[0])
        out["end_date"]   = parsed.apply(lambda t: t[1])
    else:
        out["start_date"] = None
        out["end_date"]   = None

    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


def clean_kst(df):
    """
    K-Startup DataFrame을 공통 스키마로 변환합니다.

    주요 처리:
    - pbanc_sn        : float → int → str 변환 (소수점 제거)
    - biz_trgt_age    : "만 19~39세" 텍스트 → min/max 파싱
    - supt_regin      : 이미 지역명 → 그대로 region
    - provider        : pbanc_ntrp_nm 또는 biz_prch_dprt_nm 선택
    """
    m   = COLUMN_MAP["kst"]
    out = pd.DataFrame()

    out["source"] = df["_source"]

    # source_id 소수점 제거
    out["source_id"] = df.get(m["source_id"], pd.Series([None]*len(df))).apply(fix_kst_source_id)

    out["title"] = df.get(m["title"])

    # summary = title + 원본 설명
    titles    = df.get(m["title"], pd.Series([None]*len(df)))
    summaries = df.get(m["summary_main"], pd.Series([None]*len(df)))
    out["summary"] = [build_summary(t, s) for t, s in zip(titles, summaries)]

    out["category"] = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)

    # region (K-Startup은 이미 지역명)
    out["region"] = df.get(m["region_raw"], pd.Series([DEFAULT_REGION]*len(df))).fillna(DEFAULT_REGION)
    out["region_code"] = None

    # provider (여러 후보 중 첫 번째 존재하는 값)
    provider_series = pick_first_available(df, m["provider_alts"])
    out["provider"] = provider_series.fillna(DEFAULT_PROVIDER).astype(str).replace({"nan": DEFAULT_PROVIDER, "None": DEFAULT_PROVIDER})

    out["target_group"] = df.get(m["target_group"])

    # 연령 파싱 + 유효성 검증
    age_col = m["target_age"]
    if age_col in df.columns:
        ages = df[age_col].apply(parse_kst_age)
        raw_min = ages.apply(lambda t: t[0])
        raw_max = ages.apply(lambda t: t[1])
        validated = [validate_age_range(mn, mx) for mn, mx in zip(raw_min, raw_max)]
        out["target_age_min"] = [v[0] for v in validated]
        out["target_age_max"] = [v[1] for v in validated]
    else:
        out["target_age_min"] = DEFAULT_AGE_MIN
        out["target_age_max"] = DEFAULT_AGE_MAX

    out["start_date"] = df.get(m["start_date"], pd.Series([None]*len(df))).apply(normalize_date)
    out["end_date"]   = df.get(m["end_date"],   pd.Series([None]*len(df))).apply(normalize_date)
    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


def clean_youth(df):
    """
    Youthcenter DataFrame을 공통 스키마로 변환합니다.

    주요 처리:
    - zipCd           : region_code에 원본 보존 + region에 광역 한글명
    - summary         : title + plcyExplnCn + plcySprtCn 통합
    - provider        : sprvsnInstCdNm 또는 operInstCdNm 선택
    - 연령            : ageMin/ageMax 자동 탐색 + 유효성 검증
    """
    m   = COLUMN_MAP["youth"]
    out = pd.DataFrame()

    out["source"]    = df["_source"]
    out["source_id"] = df.get(m["source_id"])
    out["title"]     = df.get(m["title"])

    # summary = title + plcyExplnCn + plcySprtCn
    titles  = df.get(m["title"], pd.Series([None]*len(df)))
    mains   = df.get(m["summary_main"], pd.Series([None]*len(df)))
    extras_col = m.get("summary_extra")
    if extras_col and extras_col in df.columns:
        extras = df[extras_col]
    else:
        extras = pd.Series([None]*len(df))
    out["summary"] = [build_summary(t, main, extra)
                      for t, main, extra in zip(titles, mains, extras)]

    out["category"] = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)

    # region: zipCd → 광역 한글명
    zipcd_series = df.get(m["region_raw"], pd.Series([None]*len(df)))
    out["region"]      = [zipcd_to_region(v) for v in zipcd_series]
    out["region_code"] = zipcd_series.astype(object).where(zipcd_series.notna(), None)

    # provider (여러 후보 중 첫 번째 존재)
    provider_series = pick_first_available(df, m["provider_alts"])
    out["provider"] = provider_series.fillna(DEFAULT_PROVIDER).astype(str).replace({"nan": DEFAULT_PROVIDER, "None": DEFAULT_PROVIDER})

    out["target_group"] = df.get(m["target_group"])

    # 연령 컬럼 자동 탐색
    min_cands = [c for c in df.columns if "min" in c.lower() and "age" in c.lower()]
    max_cands = [c for c in df.columns if "max" in c.lower() and "age" in c.lower()]
    print(f"  [youth 연령 컬럼] min 후보: {min_cands}, max 후보: {max_cands}")

    min_col = min_cands[0] if min_cands else None
    max_col = max_cands[0] if max_cands else None

    if min_col or max_col:
        min_s = df[min_col] if min_col else pd.Series([None]*len(df))
        max_s = df[max_col] if max_col else pd.Series([None]*len(df))
        raw_pairs = [parse_youth_age(mn, mx) for mn, mx in zip(min_s, max_s)]
        validated = [validate_age_range(mn, mx) for mn, mx in raw_pairs]
        out["target_age_min"] = [v[0] for v in validated]
        out["target_age_max"] = [v[1] for v in validated]
    else:
        print("  [youth 연령 컬럼] 탐색 실패 → 기본값(0~99) 적용")
        out["target_age_min"] = DEFAULT_AGE_MIN
        out["target_age_max"] = DEFAULT_AGE_MAX

    out["start_date"] = df.get(m["start_date"], pd.Series([None]*len(df))).apply(normalize_date)
    out["end_date"]   = df.get(m["end_date"],   pd.Series([None]*len(df))).apply(normalize_date)
    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


print("source별 정규화 함수 정의 완료")
print("정의된 함수: clean_bizinfo(), clean_kst(), clean_youth()")

## 1단계. 데이터 불러오기

- `USE_MOCK = True`  → mock 데이터 사용 (실제 CSV 없어도 동작)
- `USE_MOCK = False` → `combined_raw_columns.csv` 사용 (v1.0 실행 후 가능)

In [ ]:
if USE_MOCK:
    print("[USE_MOCK=True] mock 데이터를 사용합니다.")
    raw_df = pd.DataFrame(MOCK_ROWS)
    # Python None이 유지되도록 처리
    raw_df = raw_df.where(raw_df.notna(), None)
else:
    print("[USE_MOCK=False] 실제 CSV를 사용합니다.")
    if not RAW_MERGED_FILE.exists():
        raise FileNotFoundError(
            f"파일 없음: {RAW_MERGED_FILE}\n"
            "PolicyRec_v1.0.ipynb를 먼저 실행해 주세요."
        )
    # dtype=str로 읽으면 숫자가 소수점으로 변환되는 문제는 막을 수 있지만
    # 그 경우 "177322.0" 같은 문자열이 남을 수 있으므로 fix_kst_source_id에서 재변환.
    raw_df = pd.read_csv(RAW_MERGED_FILE, encoding=CSV_ENCODING, dtype=str)

print(f"불러온 데이터: {raw_df.shape[0]}행 × {raw_df.shape[1]}열")
print("\nsource별 행 개수:")
display(raw_df["_source"].value_counts().rename_axis("source").reset_index(name="row_count"))

## 2단계. 매핑 컬럼 존재 여부 사전 확인

정규화 실행 전, 매핑 대상 컬럼이 실제로 있는지 확인합니다.
`존재: ✗`인 항목은 해당 source에 컬럼이 없거나 컬럼명이 다른 경우입니다.
→ `COLUMN_MAP`에서 컬럼명을 수정하거나, 기본값 처리 방식을 조정하세요.

In [ ]:
# ============================================================
# 매핑 대상 컬럼 존재 여부 체크
# ============================================================

def check_columns(raw_df, source_code):
    """특정 source의 매핑 대상 컬럼이 실제 DataFrame에 있는지 확인"""
    m = COLUMN_MAP[source_code]
    sub = raw_df[raw_df["_source"] == source_code]
    rows = []
    for key, col_or_list in m.items():
        # 1) 리스트 타입 먼저 체크 (provider_alts 등)
        if isinstance(col_or_list, list):
            existing = [c for c in col_or_list if c in sub.columns]
            exists_str = "✓" if existing else "✗"
            non_null = sum(sub[c].notna().sum() for c in existing) if existing else 0
            col_label = " / ".join(col_or_list) + (f" (사용: {existing[0]})" if existing else "")
            rows.append({"source": source_code, "공통 컬럼": key,
                         "원본 컬럼": col_label, "존재": exists_str,
                         "non-null": int(non_null) if existing else "—"})
            continue

        # 2) None 또는 내부 토큰 (_age_range 등) → 건너뛰기
        if col_or_list is None or (isinstance(col_or_list, str) and col_or_list.startswith("_")):
            rows.append({"source": source_code, "공통 컬럼": key,
                         "원본 컬럼": "(없음/내부처리)", "존재": "—", "non-null": "—"})
            continue

        # 3) 단일 문자열 컬럼명
        col = col_or_list
        if col in sub.columns:
            rows.append({"source": source_code, "공통 컬럼": key,
                         "원본 컬럼": col, "존재": "✓",
                         "non-null": int(sub[col].notna().sum())})
        else:
            rows.append({"source": source_code, "공통 컬럼": key,
                         "원본 컬럼": col, "존재": "✗", "non-null": 0})
    return rows


all_checks = []
for src in SOURCES:
    all_checks.extend(check_columns(raw_df, src))

check_df = pd.DataFrame(all_checks)
print("=== 매핑 컬럼 존재 여부 ===")
display(check_df)

## 3단계. 정규화 실행

각 source별로 `clean_*()` 함수를 호출해 공통 스키마로 변환합니다.

In [ ]:
# ============================================================
# source별 정규화 실행
# ============================================================

CLEANERS = {
    "biz":   clean_bizinfo,
    "kst":   clean_kst,
    "youth": clean_youth,
}

normalized_frames = []
for src in SOURCES:
    sub = raw_df[raw_df["_source"] == src]
    if len(sub) == 0:
        print(f"[{src}] 데이터 없음 → 스킵")
        continue
    print(f"[{src}] {len(sub)}건 정규화 시작")
    cleaned = CLEANERS[src](sub)
    print(f"[{src}] 완료: {cleaned.shape}")
    normalized_frames.append(cleaned)

print(f"\n총 {len(normalized_frames)}개 DataFrame 준비 완료")

## 4단계. 병합 및 저장

In [ ]:
# ============================================================
# 병합 및 CSV 저장
# ============================================================

normalized_df = pd.concat(normalized_frames, ignore_index=True)

CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
normalized_df.to_csv(NORMALIZED_FILE, index=False, encoding=CSV_ENCODING)

print(f"저장 완료: {NORMALIZED_FILE}")
print(f"전체 행/열: {normalized_df.shape}")
print("\nsource별 행 개수:")
display(normalized_df["source"].value_counts().rename_axis("source").reset_index(name="row_count"))
print("\n컬럼 목록:")
print(list(normalized_df.columns))

## 5단계. 결과 미리보기

In [ ]:
# ============================================================
# source별 미리보기
# ============================================================

for src in SOURCES:
    sub = normalized_df[normalized_df["source"] == src]
    if len(sub) == 0:
        continue
    print(f"\n=== {SOURCE_LABELS[src]} ({len(sub)}건) — 상위 {PREVIEW_ROW_COUNT}건 ===")
    # 핵심 컬럼만 뽑아서 표시 (너무 넓지 않도록)
    display(sub[["source_id", "title", "region", "region_code", "provider",
                 "target_age_min", "target_age_max",
                 "start_date", "end_date"]].head(PREVIEW_ROW_COUNT))

## 6단계. 결과 검증

- 컬럼별 결측 비율
- 날짜 형식 이상값 (YYYY-MM-DD 또는 `추후공지`가 아닌 경우)
- 연령 이상값 (min > max)
- region/provider 분리 결과 요약

In [ ]:
# ============================================================
# 결과 검증
# ============================================================

DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")
ALLOWED_DATE_STRINGS = {"추후공지"}  # 날짜 아닌 허용 문자열

# ---- 결측 비율 ----
print("=== 컬럼별 결측 비율 ===")
null_df = normalized_df.isna().mean().mul(100).round(1).rename("null%").reset_index()
null_df.columns = ["column", "null%"]
display(null_df)

# ---- 날짜 형식 이상값 ----
print("\n=== 날짜 형식 이상값 (YYYY-MM-DD 또는 '추후공지' 외) ===")
for date_col in ["start_date", "end_date"]:
    def _is_valid(v):
        if pd.isna(v):
            return True  # 결측은 별도 체크
        s = str(v)
        return bool(DATE_PATTERN.match(s)) or s in ALLOWED_DATE_STRINGS

    bad_mask = ~normalized_df[date_col].apply(_is_valid)
    if bad_mask.sum():
        print(f"[{date_col}] 이상값 {bad_mask.sum()}건:")
        display(normalized_df.loc[bad_mask, ["source", "source_id", date_col]])
    else:
        print(f"[{date_col}] 이상값 없음 ✓")

# ---- 연령 이상값 ----
print("\n=== 연령 이상값 (min > max) ===")
age_bad = (
    normalized_df["target_age_min"].notna() &
    normalized_df["target_age_max"].notna() &
    (normalized_df["target_age_min"].astype(float) > normalized_df["target_age_max"].astype(float))
)
if age_bad.sum():
    display(normalized_df.loc[age_bad, ["source","source_id","target_age_min","target_age_max"]])
else:
    print("이상값 없음 ✓")

# ---- region / provider 분포 ----
print("\n=== region 분포 ===")
display(normalized_df["region"].value_counts().head(20).rename_axis("region").reset_index(name="count"))

print("\n=== provider 분포 (상위 15건) ===")
display(normalized_df["provider"].value_counts().head(15).rename_axis("provider").reset_index(name="count"))

# ---- 특수 날짜 확인 ----
print("\n=== 특수 날짜 값 확인 ===")
perpetual = (normalized_df["end_date"] == PERPETUAL_DATE).sum()
tbd       = (normalized_df["end_date"] == "추후공지").sum()
print(f"  end_date = '{PERPETUAL_DATE}' (상시): {perpetual}건")
print(f"  end_date = '추후공지'              : {tbd}건")

## 정리와 다음 작업

### v1.1 (이번 업데이트)에서 완료된 것

1. ✅ source별 정규화 함수 (`clean_bizinfo`, `clean_kst`, `clean_youth`)
2. ✅ 날짜 형식 통일 + 특수값 처리 (`상시` → `2099-12-31`, `추후공지` → 문자열 그대로)
3. ✅ 연령 파싱 + **유효성 검증** (`min > max`, `0/0` 케이스 자동 보정)
4. ✅ **region / provider 분리** (Bizinfo 부처명이 region에 섞이는 문제 해결)
5. ✅ **Youth zipCd → region 한글명 추정** + `region_code`에 원본 보존
6. ✅ **summary = title + 원본summary 통합** (youth는 plcyExplnCn + plcySprtCn)
7. ✅ **benefit_type 컬럼 삭제** (100% 결측)
8. ✅ **K-Startup source_id** float → int → str 변환 (`177322.0` → `"177322"`)

### 알려진 한계 (v1.3 이후 과제)

- **Youth 시군구 매핑**: 현재는 광역(17개)까지만 변환. 시군구(예: 제주시, 서귀포시)는 `region_code`를 재파싱해야 함.
- **행정개편 대응**: 현 매핑은 2024~2026년 기준. 특별자치도 전환 등 변경 시 `REGION_CODE_TO_NAME` 업데이트 필요.
- **Provider 정규화**: 같은 기관이라도 표기가 다를 수 있음 (예: "중기부" vs "중소벤처기업부"). 별도 정규화 단계 필요.
- **category 통일**: source별 분류 체계가 달라 그대로 병합됨. 공통 택소노미 매핑이 필요할 수 있음.

### 다음 단계: v1.2 — SQLite + Chroma

1. `combined_normalized_v1_1.csv`를 입력으로 사용
2. **SQLite DB** 생성: 구조화된 필터링용 (지역, 연령, 카테고리, 날짜)
3. **Chroma 컬렉션** 생성: 의미 기반 검색용 (title + summary 임베딩)
4. 두 저장소를 `source_id`로 연결